In [1]:
!pip install requests pandas tqdm
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.trips_data_api import fetch_with_retry, fetch_all_rides, fetch_stops_for_ride, haversine, fmt_time, fmt_date, day_he, round_hour, dur_min
import time
from tqdm import tqdm
import requests
import pandas as pd

In [4]:
BASE = "https://open-bus-stride-api.hasadna.org.il"

rides = fetch_all_rides(start_date="2024-01-07", end_date="2024-01-07")

Total rides to fetch: 19462
1% 2% 3% 4% 5% 6% 7% 8% 9% 10% 11% 12% 13% 14% 15% 16% 17% 18% 19% 20% 21% 22% 23% 24% 25% 26% 27% 28% 29% 30% 31% 32% 33% 34% 35% 36% 37% 38% 39% 40% 41% 42% 43% 44% 45% 46% 47% 48% 49% 50% 51% 52% 53% 54% 55% 56% 57% 58% 59% 60% 61% 62% 63% 64% 65% 66% 67% 68% 69% 70% 71% 72% 73% 74% 75% 76% 77% 78% 79% 80% 81% 82% 83% 84% 85% 86% 87% 88% 89% 90% 91% 92% 93% 94% 95% 96% 97% 98% 99% 100% 
✅ Total 19462 rides


In [6]:
df_raw = pd.DataFrame(rides)

In [ ]:
df_raw['gtfs_route__route_mkt'] = df_raw['gtfs_route__route_mkt'].astype(int)

In [13]:
df_raw['gtfs_route__line_ref'] = df_raw['gtfs_route__line_ref'].astype(int)

In [14]:

dictionary = df_raw.drop_duplicates(subset=['gtfs_route__line_ref']).set_index('gtfs_route__line_ref')['gtfs_route__route_mkt'].to_dict()

In [17]:
dictionary[27082]

99002

In [15]:
import json

path = r'C:\Users\shaha\Documents\DS_course\public_transport_ML\data\others\route_id_to_mkt_id.txt'

with open(path, 'w') as f:
    json.dump(dictionary, f, indent=4)

In [18]:
rides[0]

{'id': 58680875,
 'siri_route_id': 293,
 'journey_ref': '2024-01-07-584905622',
 'scheduled_start_time': '2024-01-07T00:00:00+00:00',
 'vehicle_ref': '9251301',
 'updated_first_last_vehicle_locations': '2024-01-07T03:46:33.240401+00:00',
 'first_vehicle_location_id': 3171727064,
 'last_vehicle_location_id': 3171729046,
 'updated_duration_minutes': '2024-01-07T07:05:42.965729+00:00',
 'duration_minutes': 5,
 'journey_gtfs_ride_id': None,
 'route_gtfs_ride_id': 63500636,
 'gtfs_ride_id': 63500636,
 'siri_route__line_ref': 27082,
 'siri_route__operator_ref': 5,
 'gtfs_ride__gtfs_route_id': 4128222,
 'gtfs_ride__journey_ref': '584905622_060124',
 'gtfs_ride__start_time': '2024-01-07T00:00:00+00:00',
 'gtfs_ride__end_time': '2024-01-07T00:58:59+00:00',
 'gtfs_route__date': '2024-01-07',
 'gtfs_route__line_ref': 27082,
 'gtfs_route__operator_ref': 5,
 'gtfs_route__route_short_name': '2',
 'gtfs_route__route_long_name': "פרופ' ישעיהו ליבוביץ'/הרב יעקב טראב-תל אביב יפו<->משמר הירדן/טורקוב-תל א

In [ ]:
# Build cache: one ride for each unique route
route_cache = {}
seen = {}

for ride in rides:
    rid = ride.get("gtfs_ride__gtfs_route_id")
    if rid and rid not in seen and ride.get("gtfs_ride_id"):
        seen[rid] = ride["gtfs_ride_id"]

print(f"Fetching stops for {len(seen)} unique routes...")

total = len(seen)
next_percent_to_print = 1

for idx, (route_id, gtfs_ride_id) in enumerate(seen.items()):
    try:
        stops = fetch_stops_for_ride(gtfs_ride_id)
        if not isinstance(stops, list):
            stops = []

        dist = 0
        for i in range(1, len(stops)):
            a, b = stops[i - 1], stops[i]
            if a.get("gtfs_stop__lat") and b.get("gtfs_stop__lat"):
                dist += haversine(
                    a["gtfs_stop__lat"], a["gtfs_stop__lon"],
                    b["gtfs_stop__lat"], b["gtfs_stop__lon"]
                )

        route_cache[route_id] = {
            "from_city": stops[0].get("gtfs_stop__city", "") if stops else "",
            "from_stop": stops[0].get("gtfs_stop__name", "") if stops else "",
            "to_city": stops[-1].get("gtfs_stop__city", "") if stops else "",
            "to_stop": stops[-1].get("gtfs_stop__name", "") if stops else "",
            "stop_count": len(stops),
            "dist_km": round(dist, 3),
        }

    except Exception as e:
        route_cache[route_id] = {}

    current_percent = int((idx + 1) / total * 100) if total > 0 else 100
    while next_percent_to_print <= current_percent and next_percent_to_print <= 100:
        print(f"{next_percent_to_print}%", end=' ', flush=True)
        next_percent_to_print += 1

    time.sleep(0.15)

print(f"\n✅ Cache ready: {len(route_cache)} routes")

In [ ]:
rows = []
for ride in rides:
    dep  = ride.get("gtfs_ride__start_time") or ride.get("scheduled_start_time","")
    arr  = ride.get("gtfs_ride__end_time","")
    plan = dur_min(dep, arr)
    act  = ride.get("duration_minutes") if ride.get("duration_minutes",0) > 0 else ""
    diff = (act - plan) if isinstance(plan,int) and isinstance(act,int) else ""
    s    = route_cache.get(ride.get("gtfs_ride__gtfs_route_id"), {})
    dist = s.get("dist_km", 0)
    sp_plan = round(dist / (plan/60), 1) if dist and isinstance(plan,int) and plan > 0 else ""
    sp_act  = round(dist / (act/60),  1) if dist and isinstance(act,int)  and act  > 0 else ""

    rows.append({
        "תאריך":                   fmt_date(dep),
        "יום":                     day_he(dep),
        "שעה (עגולה)":            round_hour(dep),
        "מספר קו":                 ride.get("gtfs_route__route_short_name",""),
        "שם הקו":                  ride.get("gtfs_route__route_long_name",""),
        "route_id":               ride.get("gtfs_route__line_ref",""),
        "route_mkt":              ride.get("gtfs_route__route_mkt",""),
        "כיוון":                   ride.get("gtfs_route__route_direction",""),
        "אלטרנטיבה":               ride.get("gtfs_route__route_alternative",""),
        "חברה (agency_name)":      ride.get("gtfs_route__agency_name",""),
        "סוג מסלול (route_type)":  "אוטובוס" if ride.get("gtfs_route__route_type")=="3" else ride.get("gtfs_route__route_type",""),
        "עיר מוצא":                s.get("from_city",""),
        "תחנת מוצא":               s.get("from_stop",""),
        "עיר יעד":                 s.get("to_city",""),
        "תחנת יעד":                s.get("to_stop",""),
        "כמות תחנות":              s.get("stop_count",""),
        "אורך מסלול (קמ)":         dist or "",
        "זמן יציאה מתוכנן":        fmt_time(dep),
        "זמן הגעה מתוכנן":         fmt_time(arr),
        "משך מתוכנן (דק)":         plan,
        "משך בפועל (דק)":          act,
        "הפרש (דק)":               diff,
        "מהירות מתוכננת (קמש)":    sp_plan,
        "מהירות בפועל (קמש)":      sp_act,
        "מזהה SIRI":               ride.get("id",""),
    })

df = pd.DataFrame(rows)

In [ ]:
df.to_csv("../data/raw_data/entire_data/telaviv_buses_0501_1101_2024.csv", index=False, encoding="utf-8-sig")
print(f"✅ נשמר: telaviv_buses_1201_1801_2024.csv")
print(df.shape)
df.head(3)

In [ ]:
df_2 = pd.read_csv("../data/raw_data/entire_data/telaviv_buses_1201_1801_2024.csv")

In [ ]:
pd.concat([df, df_2], ignore_index=True).to_csv("../data/raw_data/entire_data/telaviv_buses_0501_1801_2024.csv", index=False, encoding="utf-8-sig")

In [ ]:
df[(df['חברה (agency_name)'] == 'דן') & (df['מספר קו'] == '4')]